<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Lectures/CNN%20Lab%20with%20Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CNN Lab with Keras**

## **Table of Contents**

1. [Pre-Lab Preparation](#section0)  
2. [Introduction & Concepts](#section1)  
   - 2.1 [Neural Network Fundamentals](#section1_1)  
   - 2.2 [Image Fundamentals & Manual Convolution Demo](#section1_2)  
   - 2.3 [Why CNNs vs. Dense Networks?](#section1_3)  
   - 2.4 [Environment & Dataset Setup Tips](#section1_4)  
3. [Part I: Comparing Dense Network vs. CNN (MNIST)](#section2)  
   - 3.1 [Overview & Time Estimate](#section2_1)  
   - 3.2 [Build & Train a Dense Network (Baseline)](#section2_2)  
   - 3.3 [Build & Train a CNN on MNIST](#section2_3)  
   - 3.4 [Visualizing Learned Features on MNIST](#section2_4)  
   - 3.5 [Try-It-Yourself (TIY)](#section2_5)  
   - 3.6 [Reflection & Summary (MNIST)](#section2_6)  
4. [Part II: Data Augmentation with CIFAR-10](#section3)  
   - 4.1 [Overview & Time Estimate](#section3_1)  
   - 4.2 [Demonstration of Data Augmentation](#section3_2)  
   - 4.3 [Implementing a CNN with Augmentation](#section3_3)  
   - 4.4 [Try-It-Yourself (TIY)](#section3_4)  
   - 4.5 [Reflection & Summary (CIFAR-10)](#section3_5)  
5. [Part III: Transfer Learning (Cats vs. Dogs)](#section4)  
   - 5.1 [What is Transfer Learning?](#section4_1)  
   - 5.2 [Feature Extraction vs. Fine-Tuning](#section4_2)  
   - 5.3 [Try-It-Yourself (TIY)](#section4_3)  
   - 5.4 [Reflection & Summary (Transfer Learning)](#section4_4)  
6. [TensorBoard Demonstration](#section5)  
7. [Additional Best Practices](#section6)  
8. [Common Pitfalls & Tips](#section7)  
9. [Wrap-Up & Further Resources](#section8)  
10. [Mini Project & Optional Extensions](#section9)  
11. [(Optional) Sample Solutions to TIY](#section10)

<a name="section0"></a>

## 1. **Pre-Lab Preparation**

1. **Read Basic CNN Materials**  
   - E.g., short notes from Andrew Ng’s course or a summary of convolution/pooling operations.  
2. **Install/Verify Environment**  
   - Python 3.7+, TensorFlow, NumPy, Matplotlib, etc.  
   - Confirm you can import them: `import tensorflow as tf`, `import numpy as np`, etc.  
3. **Datasets**  
   - MNIST and CIFAR-10 load automatically with `tf.keras.datasets`.  
   - Cats vs. Dogs might require manual setup in the `data/` directory structure.  
4. **GPU**  
   - Use a GPU for faster training if possible (local or Google Colab).  

<a name="section1"></a>

## 2. **Introduction & Concepts**

<a name="section1_1"></a>
### 2.1 Neural Network Fundamentals

- **Neuron**: $\text{activation}(\mathbf{w} \cdot \mathbf{x} + b)$.  
- **Layers**: Dense (fully-connected), Convolutional, Pooling.  
- **Forward Pass & Backpropagation**: Minimizing a loss function by adjusting weights.

<a name="section1_2"></a>
### 2.2 Image Fundamentals & Manual Convolution Demo

Images are 2D grids of pixels (or 3D with RGB). Convolutions slide filters to detect local patterns like edges.

**Manual Convolution Example**:

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import random
from PIL import Image
from scipy.signal import convolve2d

# Reproducibility
random.seed(123)
np.random.seed(123)
tf.random.set_seed(123)

img = Image.open('sample_image.jpg').convert('L')  # grayscale
img_array = np.array(img)

vertical_filter = np.array([[-1, 0, 1],
                            [-1, 0, 1],
                            [-1, 0, 1]])

horizontal_filter = np.array([[ 1,  1,  1],
                              [ 0,  0,  0],
                              [-1, -1, -1]])

vertical_edges   = convolve2d(img_array, vertical_filter, mode='same', boundary='symm')
horizontal_edges = convolve2d(img_array, horizontal_filter, mode='same', boundary='symm')

fig, axs = plt.subplots(1, 3, figsize=(16,5))
axs[0].imshow(img_array, cmap='gray')
axs[0].set_title("Original")
axs[0].axis('off')

axs[1].imshow(vertical_edges, cmap='gray')
axs[1].set_title("Vertical Edges")
axs[1].axis('off')

axs[2].imshow(horizontal_edges, cmap='gray')
axs[2].set_title("Horizontal Edges")
axs[2].axis('off')

plt.tight_layout()
plt.show()

<a name="section1_3"></a>
### 2.3 Why CNNs vs. Dense Networks?

- **Dense Networks**: Flatten images → loses spatial structure, large parameter count.  
- **CNNs**: Exploit 2D structure via filters, fewer parameters, typically better performance on images.

<a name="section1_4"></a>
### 2.4 Environment & Dataset Setup Tips

- **Directory Structure**: Keep your `.ipynb` or `.py` files and `data` folder in the same place for easy loading.  
- **GPU vs. CPU**: CNN training can be slow on CPU—prefer GPU if available.  
- **Reproducibility**: Setting seeds (`tf.random.set_seed(...)`) can help, though GPU concurrency might still introduce minor differences.


<a name="section2"></a>

## 3. **Part I: Comparing Dense Network vs. CNN (MNIST)**

<a name="section2_1"></a>
### 3.1 Overview & Time Estimate

- **Dataset**: 70,000 images (28×28) of handwritten digits (0-9).  
- **Expected Time**: ~15 minutes for reading & coding, plus a few more for training on a GPU or ~20+ mins on CPU (depending on hardware).

<a name="section2_2"></a>
### 3.2 Build & Train a Dense Network (Baseline)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# Flatten
x_train_flat = x_train.reshape(-1, 28*28)
x_test_flat  = x_test.reshape(-1, 28*28)

model_dense = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_dense.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

history_dense = model_dense.fit(
    x_train_flat, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

test_loss_dense, test_acc_dense = model_dense.evaluate(x_test_flat, y_test, verbose=0)
print(f"Dense Network Test Accuracy: {test_acc_dense:.4f}")

<a name="section2_3"></a>
### 3.3 Build & Train a CNN on MNIST

In [ ]:
# Reshape for CNN
x_train_cnn = np.expand_dims(x_train, axis=-1)
x_test_cnn  = np.expand_dims(x_test, axis=-1)

model_cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

history_cnn = model_cnn.fit(
    x_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

test_loss_cnn, test_acc_cnn = model_cnn.evaluate(x_test_cnn, y_test, verbose=0)
print(f"CNN Test Accuracy: {test_acc_cnn:.4f}")

<a name="section2_4"></a>
### 3.4 Visualizing Learned Features on MNIST

#### 3.4.1 Visualizing Filters

In [ ]:
filters, biases = model_cnn.layers[0].get_weights()
print("Filter shape:", filters.shape)  # e.g., (3,3,1,32)

import math

num_filters = filters.shape[-1]
n_cols = 8
n_rows = math.ceil(num_filters / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12,12))

for i in range(num_filters):
    row = i // n_cols
    col = i % n_cols
    f = filters[:, :, 0, i]
    axes[row, col].imshow(f, cmap='gray')
    axes[row, col].axis('off')

plt.suptitle("First Conv2D Filters (MNIST)", fontsize=14)
plt.tight_layout()
plt.show()

#### 3.4.2 Visualizing Layer Activations

In [ ]:
from tensorflow.keras import models

layer_outputs = [layer.output for layer in model_cnn.layers[:6]]  # up to Flatten
activation_model = models.Model(inputs=model_cnn.input, outputs=layer_outputs)

sample_img = x_test_cnn[0:1]
activations = activation_model.predict(sample_img)

for layer_idx, layer_activation in enumerate(activations):
    num_features = layer_activation.shape[-1]
    n_cols = 8
    n_rows = math.ceil(num_features / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12,12))
    for i in range(num_features):
        row = i // n_cols
        col = i % n_cols
        channel_img = layer_activation[0, :, :, i]
        axes[row, col].imshow(channel_img, cmap='gray')
        axes[row, col].axis('off')

    plt.suptitle(f"Layer {layer_idx} Activation Maps")
    plt.tight_layout()
    plt.show()

<a name="section2_5"></a>
### 3.5 Try-It-Yourself (TIY)

#### **TIY #1: Changing Kernel Size in CNN**

In [ ]:
# === TIY: Modify kernel size in your CNN ===
model_cnn_tiy = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (# Insert Your Code Here ), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (# Insert Your Code Here ), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_cnn_tiy.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Insert Your Code Here:
#   1. Fit model_cnn_tiy with (x_train_cnn, y_train), 5 epochs, batch_size=64
#   2. Evaluate on (x_test_cnn, y_test)
#   3. Print the test accuracy

**Hint**: Try `(5,5)` or `(7,7)`. Check `model_cnn_tiy.summary()` to see how parameter counts change.

#### **TIY #2: Visualize Misclassifications**

In [ ]:
# === TIY: Compare misclassifications in Dense vs. CNN ===

# Insert Your Code Here:
# 1. preds_dense = model_dense.predict(x_test_flat)
# 2. preds_cnn   = model_cnn.predict(x_test_cnn)
# 3. Convert to class indices with np.argmax(...)
# 4. Find indexes where each model is wrong, plot some examples

#### **TIY #3: Add/Remove a Convolutional Layer**

In [ ]:
# === TIY: Architectural Change ===

# Insert Your Code Here:
# 1. Build a new CNN with one extra Conv2D -> MaxPooling2D block
# 2. Train & evaluate
# 3. Compare results to original

<a name="section2_6"></a>
### 3.6 Reflection & Summary (MNIST)

1. **Which** model performed better, dense or CNN?  
2. **How** does kernel size or layer depth affect performance & training time?  
3. **Observations** from the activation maps?

Short Quiz:
1. *What does MaxPooling2D do?*  
2. *If you see overfitting on MNIST, how might you fix it?* (e.g. add dropout, data augmentation, or reduce capacity)

<a name="section3"></a>

## 4. **Part II: Data Augmentation with CIFAR-10**

<a name="section3_1"></a>
### 4.1 Overview & Time Estimate

- **Dataset**: CIFAR-10 (60,000 color images, 32×32).  
- **Time**: ~25–30 minutes (demo + training on GPU). Possibly more on CPU.

<a name="section3_2"></a>
### 4.2 Demonstration of Data Augmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

(x_train_c10, y_train_c10), (x_test_c10, y_test_c10) = tf.keras.datasets.cifar10.load_data()
x_train_c10 = x_train_c10.astype('float32') / 255.0
x_test_c10  = x_test_c10.astype('float32')  / 255.0

sample_image = x_train_c10[0]

demo_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

sample_batch = np.expand_dims(sample_image, axis=0)
aug_iter = demo_datagen.flow(sample_batch, batch_size=1)

fig, axes = plt.subplots(1, 5, figsize=(16,3))
for i in range(5):
    aug_img = next(aug_iter)[0].astype('float32')
    axes[i].imshow(aug_img)
    axes[i].axis('off')
plt.suptitle("Data Augmentation Demo (CIFAR-10)", fontsize=14)
plt.tight_layout()
plt.show()

<a name="section3_3"></a>
### 4.3 Implementing a CNN with Augmentation

In [ ]:
model_cifar = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_cifar.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train_c10)

batch_size = 64
train_generator = datagen.flow(x_train_c10, y_train_c10, batch_size=batch_size)

history_cifar = model_cifar.fit(
    train_generator,
    steps_per_epoch=len(x_train_c10)//batch_size,
    epochs=10,
    validation_data=(x_test_c10, y_test_c10)
)

test_loss_cifar, test_acc_cifar = model_cifar.evaluate(x_test_c10, y_test_c10, verbose=0)
print(f"CIFAR-10 Test Accuracy with Augmentation: {test_acc_cifar:.4f}")

<a name="section3_4"></a>
### 4.4 Try-It-Yourself (TIY)

#### **TIY #4: Visualizing More Augmented Images**

In [ ]:
# === Insert Your Code Here:
# 1. Pick a different image from x_train_c10 (e.g., x_train_c10[10])
# 2. Create or reuse ImageDataGenerator with various transforms
# 3. Plot 5 augmented versions side by side

#### **TIY #5: Adjust Augmentation Settings**

In [ ]:
# === Insert Your Code Here:
# 1. Modify rotation_range, zoom_range, etc.
# 2. Re-train
# 3. Compare accuracy vs. the default settings

#### **TIY #6: Confusion Matrix**

In [ ]:
# === Insert Your Code Here:
# 1. Predict on x_test_c10
# 2. Use sklearn.metrics.confusion_matrix
# 3. Optionally plot a heatmap

**Reflection**: Which classes are commonly confused?

<a name="section3_5"></a>
### 4.5 Reflection & Summary (CIFAR-10)

- **How** does augmentation improve generalization?  
- **Which** specific transforms helped or hurt?  
- **Quiz**: “What might happen if you apply extremely large shifts or rotations?”

<a name="section4"></a>

## 5. **Part III: Transfer Learning (Cats vs. Dogs)**

<a name="section4_1"></a>
### **5.1 What is Transfer Learning?**
- **Feature Extraction**: Use a pretrained CNN (often trained on ImageNet) as a **fixed** feature extractor. Only train a new classifier on top.  
- **Fine-Tuning**: Optionally unfreeze some upper (deeper) layers of the pretrained model to adapt more specialized features to your new task.

This is especially useful when:
- Your dataset is relatively **small** (fewer images).  
- You want to leverage powerful **general** features learned from a **large** dataset like ImageNet.

### **5.2 Dataset: Cats vs. Dogs**

#### **Option A: Kaggle Approach**
1. **Download** the dataset from the [Kaggle “Dogs vs. Cats” competition](https://www.kaggle.com/c/dogs-vs-cats).  
2. **Unzip** and **manually split** the images into subfolders for training, validation, and optionally testing. For example:
   ```
   cats_and_dogs/
     train/
       cats/
         cat.0.jpg
         cat.1.jpg
         ...
       dogs/
         dog.0.jpg
         dog.1.jpg
         ...
     validation/
       cats/
         cat.2000.jpg
         ...
       dogs/
         dog.2000.jpg
         ...
     test/
       cats/
       dogs/
   ```
3. **Use** `flow_from_directory(...)` with the paths pointing to these folders. Keras automatically infers class labels from folder names (i.e., `cats` vs. `dogs`).

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "cats_and_dogs/train"
valid_dir = "cats_and_dogs/validation"
# Optionally test_dir = "cats_and_dogs/test"

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    rescale=1./255
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)
valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

#### **Option B: TensorFlow Datasets (tfds)**
1. **Install** and import TensorFlow Datasets:

In [ ]:
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import tensorflow as tf

2. **Load** the “cats_vs_dogs” dataset:

In [ ]:
(ds_full, ds_info) = tfds.load(
       'cats_vs_dogs',
       with_info=True,
       as_supervised=True  # returns (image, label)
   )

- This dataset is around 23k images, labeled 0 for cat and 1 for dog.

3. **Split** into train, validation, and test:

In [ ]:
# For example, 80% train, 10% val, 10% test
ds_train = tfds.load('cats_vs_dogs', split='train[:80%]', as_supervised=True)
ds_valid = tfds.load('cats_vs_dogs', split='train[80%:90%]', as_supervised=True)
ds_test  = tfds.load('cats_vs_dogs', split='train[90%:]', as_supervised=True)

# Or combine into one load call with multiple splits

4. **Preprocess** (resize and normalize):

In [ ]:
def format_image(image, label):
  image = tf.image.resize(image, (224,224))
  image = image / 255.0
  return image, label

ds_train = ds_train.map(format_image).shuffle(1000).batch(32)
ds_valid = ds_valid.map(format_image).batch(32)
ds_test  = ds_test.map(format_image).batch(32)

5. **Train**:

In [ ]:
# Later, you'll pass ds_train, ds_valid directly to model.fit(...)

> **Note**: The Kaggle version and the TFDS version both have images of cats & dogs, but the sets may not be identical. The TFDS version is also curated from the same source but might differ slightly in image count/filenames.

<a name="section4_2"></a>
### 5.3 Feature Extraction vs. Fine-Tuning

Now that we have our data, we’ll demonstrate feature extraction and optional fine-tuning using a pretrained model (e.g., **VGG16** or **MobileNetV2**).

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras import layers, models

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

for layer in base_model.layers:
    layer.trainable = False  # freeze all layers initially

model_tl = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # Binary classification: cat vs dog
])

model_tl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model_tl.summary()

1. **If Using Kaggle + flow_from_directory**  

In [ ]:
history_tl = model_tl.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator
)

2. **If Using TFDS**  

In [ ]:
 history_tl = model_tl.fit(
     ds_train,  # already batched & preprocessed
     epochs=5,
     validation_data=ds_valid
)

**Check** your final accuracy. You might already get good results because the ImageNet features transfer well to distinguishing cats vs. dogs.

### **5.4 Fine-Tuning**

**Goal**: Unfreeze the **last few layers** of the base model to adapt specialized features.

In [ ]:
# Example: unfreeze last 4 layers of the base model
for layer in base_model.layers[-4:]:
    layer.trainable = True

# Recompile with smaller LR
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_ft = model_tl.fit(
    # same approach
    train_generator if using flow_from_directory else ds_train,
    epochs=5,
    validation_data=valid_generator if using flow_from_directory else ds_valid
)

**Why smaller LR?**  
- The pretrained weights are already in a good feature space. Large updates might destroy them (the “catastrophic forgetting” problem).

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    preprocessing_function=preprocess_input
)
valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    'data/train',
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)
valid_generator = valid_datagen.flow_from_directory(
    'data/validation',
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

history_tl = model_tl.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator
)

# Fine-Tuning
for layer in base_model.layers[-4:]:
    layer.trainable = True

model_tl.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

history_ft = model_tl.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator
)

### **5.5 Evaluation & Exploration**

1. **Test** or **ds_test** evaluation:

In [ ]:
# Kaggle approach:
# test_loss, test_acc = model_tl.evaluate(test_generator)
# TFDS approach:
# test_loss, test_acc = model_tl.evaluate(ds_test)

print(f"Test Accuracy: {test_acc:.4f}")

2. **Misclassifications**:
   - If using flow_from_directory, get predictions with `model_tl.predict(test_generator)`. Compare to `test_generator.classes`.
   - If using TFDS, do:
     ```python
     import numpy as np
     true_labels = []
     imgs = []

     for img_batch, label_batch in ds_test:
         preds = model_tl.predict(img_batch)
         # gather
     ```

3. **Confusion Matrix** (if you have separate sets of cat/dog images).

<a name="section4_3"></a>
### 5.3 Try-It-Yourself (TIY)

#### **TIY #7: Freeze vs. Unfreeze**

In [ ]:
# === Insert Your Code Here:
# 1. Freeze all base_model layers -> Train 3-5 epochs
# 2. Unfreeze last 4 layers -> set smaller LR -> Re-train
# 3. Compare final accuracy vs. just feature extraction

#### **TIY #8: Change Pretrained Model**

In [ ]:
# === Insert Your Code Here:
# from tensorflow.keras.applications import ResNet50
# base_model_2 = ResNet50(weights='imagenet', include_top=False, ...)
# ...

#### **TIY #9: Visualize Misclassifications**

In [ ]:
# === Insert Your Code Here:
# 1. Predict on valid_generator
# 2. Identify misclassified images
# 3. Plot some examples

<a name="section4_4"></a>
### 5.4 Reflection & Summary (Transfer Learning)

- **Did** fine-tuning help? Or did it cause overfitting?  
- **Which** pretrained model is fastest or yields best accuracy?  
- **Why** freeze earlier layers?

Quiz:
1. *Which layers typically contain more general features?* (earlier)  
2. *What’s a reason for using a smaller LR during fine-tuning?*

<a name="section5"></a>
## 6. **TensorBoard Demonstration**

**Why TensorBoard?**  
- Track training/validation loss & accuracy in real time.  
- Visualize histograms of weights, activation distributions, etc.

In [ ]:
# Example usage of TensorBoard

import datetime
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_cnn_tb = model_cnn.fit(
    x_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    callbacks=[tensorboard_callback]  # pass the callback here
)

# Then in your terminal or notebook:
# %load_ext tensorboard
# %tensorboard --logdir logs/fit

<a name="section6"></a>
## 7. **Additional Best Practices**

- **Regularization**: Dropout, L2 weight decay, data augmentation.  
- **Hyperparameter Tuning**: Adjust LR, batch size, epochs, optimizer.  
- **Experiment Logging**: Keep track of model configs & results.  
- **Consistent Coding Style**: Use `tf.keras` or a uniform import approach for clarity.

<a name="section7"></a>
## 8. **Common Pitfalls & Tips**

1. **Overfitting**: Validation accuracy plateaus or decreases while training accuracy soars → use regularization, data augmentation, or smaller architecture.  
2. **Mismatch in Input Shape**: Always confirm `(height, width, channels)`.  
3. **Data Leakage**: Keep separate sets for train/validation/test.  
4. **Hardware Constraints**: Large networks can be slow on CPU—use GPU if possible.

<a name="section8"></a>
## 9. **Wrap-Up & Further Resources**

- **Key Takeaways**:  
  1. CNNs exploit spatial structure better than dense networks.  
  2. Data augmentation helps generalize.  
  3. Transfer learning is powerful for small datasets.  
  4. Visualizing filters & activations shows *what* the model learns.  

- **Explore Further**:  
  - Advanced architectures (ResNet, Inception, EfficientNet).  
  - Grad-CAM for interpretability.  
  - Object detection (YOLO, Faster R-CNN) or segmentation (U-Net).

### Additional Resources
- [Keras Official Docs](https://keras.io/)  
- [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)  
- [Stanford CS231n](http://cs231n.stanford.edu/)  
- [Andrew Ng’s Deep Learning Specialization](https://www.coursera.org/specializations/deep-learning)


<a name="section9"></a>
## 10. **Mini Project & Optional Extensions**

1. **Mini Project**  
   - **Task**: Pick a small dataset (e.g., sign-language digits, a custom Kaggle subset).  
   - **Steps**:
     - Preprocess & visualize the dataset.  
     - Build a CNN with data augmentation.  
     - (Optional) If data is limited, apply transfer learning.  
     - Evaluate & analyze misclassifications, confusion matrix, etc.  
   - **Goal**: Reinforce the entire pipeline from data loading → model training → evaluation → interpretation.

2. **Extensions**  
   - **Grad-CAM**: See which regions of an image the CNN focuses on.  
   - **AutoAugment/RandAugment**: More advanced augmentation strategies.  
   - **Hyperparameter Search**: Systematically tune LR, batch size, etc.